# Catalan's constant $G$: historical constructions and new result

Each section constructs a CMF, walks a trajectory with ramanujantools and prints $\delta_N$ at the deepest sample, where $|G-p_N/q_N|=q_N^{-(1+\delta_N)}$ for the reduced $p_N/q_N$. Run the cells in order with `cmfs.py` next to this notebook. A finite-depth $\delta_N$ does not prove a measure bound.

In [1]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 matplotlib==3.11.2

## Setup

$(s_0,s_1,\ldots)$ is the first column of the walk.

In [2]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx
from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

from cmfs import eskandari_catalan, zudilin_catalan_seeds

sys.set_int_max_str_digits(0)
with ctx.workdps(162_000):  # q_N reaches about 80,000 digits at depth 3000
    CATALAN = arb.const_catalan()
DEPTHS = list(range(10, 3001, 10))
x0, x1, x2, x3, y0, y1, y2 = sp.symbols("x0 x1 x2 x3 y0 y1 y2")
results = {}
accuracy = {}


def delta(approximant):
    """delta with |G - p/q| = q^-(1 + delta), for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):  # enough bits to resolve |G - p/q|
        error = abs(CATALAN - arb(p) / q)
        assert error > 0
        return float((-1 - error.log() / arb(q).log()).mid())

def correct_digits(approximant):
    """D_N = -log10 |G - p/q|, for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):
        error = abs(CATALAN - arb(p) / q)
        assert error > 0
        return float((-error.log() / arb(10).log()).mid())

## 1. Zudilin (2002 preprint / 2003 publication): original construction

[An Apéry-like difference equation for Catalan's constant, Theorem 3](https://arxiv.org/pdf/math/0201024)
gives $r_N=u_NG-v_N=C_NF_N$, with
$$F_N={}_3F_2\!\left(\begin{matrix}N+1,N+1,N+\tfrac12\\2N+\tfrac32,2N+\tfrac32\end{matrix};1\right),
\quad C_N=\frac{(-1)^N}{4}B(N+\tfrac12,N+1)^2.$$
The initial coefficients are $(u_0,u_1)=(1,7/4)$ and $(v_0,v_1)=(0,13/8)$.
Start $(2,2,\tfrac32;\tfrac72,\tfrac72)$; exact readout
$$\frac{p_N}{q_N}=\frac{26w_0-59w_1}{28w_0-66w_1}.$$
This is the historical direction, distinct from the direction-search trajectory below.

Both constructions use the full rank-two, five-axis field `pFq(3, 2, 1)`;
the axes are $(x_0,x_1,x_2;y_0,y_1)$ and the direction is $(1,1,1;2,2)$.
The start is source index $N=1$, so $W_N$ is a walk of **$N-1$ steps**.
Write $(w_0,w_1)^T=W_Ne_0$.

The published growth is $u_N^{1/N}\to\varphi^5$ and
$|u_NG-v_N|^{1/N}\to\varphi^{-5}$, where $\varphi=(1+\sqrt5)/2$.
[Theorems 1–2 of the October 2002 paper](https://arxiv.org/pdf/math/0210423)
give the denominator clearing factor $2^{4N+o(N)}D_{2N-1}^2$,
where $D_m=\operatorname{lcm}(1,\ldots,m)$.
Consequently, for reduced $p_N/q_N=v_N/u_N$,
$$\liminf\delta_N\ge\frac{2L}{L+4\log2+4}-1
=-0.4757268902208836\ldots,\qquad L=5\log\varphi.$$
This is a bound derived from the published estimates, not a decimal quoted
in the paper or an irrationality-measure bound. Reduction can improve the
exponent, and the finite-depth values need not equal this lower bound.


In [5]:
cmf = pFq(3, 2, 1)
start = {x0: 2, x1: 2, x2: sp.Rational(3, 2), y0: sp.Rational(7, 2), y1: sp.Rational(7, 2)}
trajectory = {x0: 1, x1: 1, x2: 1, y0: 2, y1: 2}
depths = list(range(10, 1001, 10))  # source indices
P, Q = zudilin_catalan_seeds("original")
walks = cmf.walk(trajectory, [N - 1 for N in depths], start)
approximants = [(P * W)[0] / (Q * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Zudilin (2002), original"] = (depths, deltas)
accuracy["Zudilin (2002), original"] = [correct_digits(r) for r in approximants]
print(f"delta at source index 1000: {deltas[-1]:.15f}")


delta at source index 1000: -0.473017704448925


## 2. Zudilin (October 2002): modified construction

[A few remarks on linear forms involving Catalan's constant, Theorem 2](https://arxiv.org/pdf/math/0210423)
gives $\widetilde r_N=\widetilde u_NG-\widetilde v_N=\widetilde C_N\widetilde F_N$, with
$$\widetilde F_N={}_3F_2\!\left(\begin{matrix}N,N,N-\tfrac12\\2N+\tfrac12,2N+\tfrac12\end{matrix};1\right),$$
$$\widetilde C_N=\frac{(-1)^{N-1}N}{2}B(N-\tfrac12,N+1)B(N,N+\tfrac12).$$
The initial coefficients are $(\widetilde u_0,\widetilde u_1)=(0,6)$ and
$(\widetilde v_0,\widetilde v_1)=(-1,5)$.
Start $(1,1,\tfrac12;\tfrac52,\tfrac52)$; same full five-axis field and direction as above,
with exact readout
$$\frac{p_N}{q_N}=\frac{10w_0-13w_1}{12w_0-14w_1}.$$
The same derived asymptotic lower bound $\liminf\delta_N\ge-0.4757268902208836\ldots$ applies.

`local_checks/verify_zudilin_catalan.py` checks both source recurrences symbolically,
the normalization at indices 1–9, exact rational approximants at 1–9 and
10,20,…,1000, all ten axis pairs at two regular points, and overlapping
Arb intervals at two precisions at source index 1000.


In [7]:
cmf = pFq(3, 2, 1)
start = {x0: 1, x1: 1, x2: sp.Rational(1, 2), y0: sp.Rational(5, 2), y1: sp.Rational(5, 2)}
trajectory = {x0: 1, x1: 1, x2: 1, y0: 2, y1: 2}
depths = list(range(10, 1001, 10))  # source indices
P, Q = zudilin_catalan_seeds("modified")
walks = cmf.walk(trajectory, [N - 1 for N in depths], start)
approximants = [(P * W)[0] / (Q * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Zudilin (2002), modified"] = (depths, deltas)
accuracy["Zudilin (2002), modified"] = [correct_digits(r) for r in approximants]
print(f"delta at source index 1000: {deltas[-1]:.15f}")


delta at source index 1000: -0.472536570512580


## 3. Nesterenko (2016)

Proc. Steklov Inst. Math. **292** (2016)

${}_3F_2(1)$ · start $(1,1,\tfrac32;\tfrac52,\tfrac52)$ · trajectory $(1,1,1;2,2)$ · $p_N/q_N=(4s_0-5s_1)/(4s_0-6s_1)$

In [3]:
cmf = pFq(3, 2, 1)
start = {x0: 1, x1: 1, x2: sp.Rational(3, 2), y0: sp.Rational(5, 2), y1: sp.Rational(5, 2)}
trajectory = {x0: 1, x1: 1, x2: 1, y0: 2, y1: 2}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(4 * W[0, 0] - 5 * W[1, 0]) / (4 * W[0, 0] - 6 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Nesterenko (2016)"] = (DEPTHS, deltas)
accuracy["Nesterenko (2016)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: -0.474009495


## 4. Direction search in ${}_3F_2(1)$

${}_3F_2(1)$ · start $(2,2,\tfrac32;\tfrac72,\tfrac72)$ · trajectory $(20,22,25;40,42)$ · $p_N/q_N=(26s_0-59s_1)/(28s_0-66s_1)$

In [4]:
cmf = pFq(3, 2, 1)
start = {x0: 2, x1: 2, x2: sp.Rational(3, 2), y0: sp.Rational(7, 2), y1: sp.Rational(7, 2)}
trajectory = {x0: 20, x1: 22, x2: 25, y0: 40, y1: 42}
depths = list(range(10, 1501, 10))  # q_N reaches about 105,000 digits at depth 1500
walks = cmf.walk(trajectory, depths, start)
approximants = [(26 * W[0, 0] - 59 * W[1, 0]) / (28 * W[0, 0] - 66 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Direction search in 3F2(1)"] = (depths, deltas)
accuracy["Direction search in 3F2(1)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 1500: {deltas[-1]:.9f}")

delta at depth 1500: -0.431308830


## 5. New best trajectory

${}_4F_3(1)$, dual · start $(4,4,-\tfrac12,0;\tfrac72,\tfrac72,4)$ · trajectory $(11,9,-2,-11;-1,0,2)$ · $p_N/q_N=(4915s_1+754s_2)/(6930s_1+1260s_2)$

The paper proves $\liminf_N\delta_N>-0.295976878$ for this trajectory (error power
$0.704023122$); the gap to the numerical value below needs further common factors
of $p_N$ and $q_N$.

In [5]:
cmf = pFq(4, 3, 1).dual()  # inverse-transpose matrices
start = {x0: 4, x1: 4, x2: -sp.Rational(1, 2), x3: 0, y0: sp.Rational(7, 2), y1: sp.Rational(7, 2), y2: 4}
trajectory = {x0: 11, x1: 9, x2: -2, x3: -11, y0: -1, y1: 0, y2: 2}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(4915 * W[1, 0] + 754 * W[2, 0]) / (6930 * W[1, 0] + 1260 * W[2, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["New best trajectory"] = (DEPTHS, deltas)
accuracy["New best trajectory"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: -0.274977139


## 6. Eskandari (2026)

[arXiv:2609.26354](https://arxiv.org/html/2609.26354v1) · on $(3,10)$, $\liminf_N\delta_N\ge-0.3768$

Rank-8 CMF of the binomial transforms $\sum_i\binom mi(-16)^iu_{k+i}$ of Eskandari's sequences (`eskandari_catalan`) · start $(0,0)$ · trajectory $(3,10)$ · $p_N/q_N=(V_0W)_0/(U_0W)_0$, with $U_0,V_0$ the first eight terms of $u,v$

In [6]:
k, m = sp.symbols("k m")
cmf = eskandari_catalan()
start = {k: 0, m: 0}
trajectory = {k: 3, m: 10}
depths = list(range(10, 101, 10))
walks = cmf.walk(trajectory, depths, start)
R = sp.Rational
U0 = Matrix([[8, -14, R(649, 8), R(-19471, 32), R(10439881, 2048), R(-372106183, 8192),
              R(55182648049, 131072), R(-2102590888447, 524288)]])  # u_0, ..., u_7
V0 = Matrix([[0, -13, R(10699, 144), R(-8025653, 14400), R(210854706467, 45158400),
              R(-67638995691893, 1625702400), R(1213719058519951259, 3147359846400),
              R(-7815506292703811398013, 2127615256166400)]])  # v_0, ..., v_7
approximants = [(V0 * W)[0] / (U0 * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Eskandari (2026)"] = (depths, deltas)
accuracy["Eskandari (2026)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 100: {deltas[-1]:.9f}")

delta at depth 100: -0.373117788


## 7. Ramanujan Machine (2020–21)

The [Catalan catalogue](https://ramanujanmachine.com/wp-content/uploads/2020/06/catalan.pdf)
lists 22 conjectured continued fractions.  They form two polynomial CMFs,
derived below.  The catalogue's last column is digits per term, not
$1+\delta$.  The [Nature paper](https://www.nature.com/articles/s41586-021-03229-4)
also gives eight examples in Supplementary Tables 5–6; these are kept
separate from the 22 catalogue rows.

For each catalogue row, $a_0+b_1/(a_1+b_2/(a_2+\cdots))=K/(cG+d)$.
The polynomials and proposed readouts are transcribed below.


In [ ]:
# Catalogue data: (a_n, b_n, K, c, d, digits/term).
# The 22 infinite-limit identities are conjectural.
n = sp.symbols("n", integer=True, nonnegative=True)
rm_rows = [
 (3*n**2+3*n+1, -2*n**4, 1, 2, 0, .29800),
 (3*n**2+3*n+1, -2*n**3*(n+1), 2, 2, -1, .28903),
 (3*n**2+3*n+1, -2*n**3*(n+2), 24, 18, -11, .27984),
 (3*n**2+3*n+1, -2*n**3*(n+3), 720, 450, -299, .27042),
 (3*n**2+7*n+3, -2*n**3*(n+2), 2, 2, -1, .30387),
 (3*n**2+7*n+3, -2*n**2*(n+1)*(n+2), 4, 2, 1, .29507),
 (3*n**2+7*n+3, -2*n**2*(n+2)**2, 16, 6, -1, .28608),
 (3*n**2+7*n+3, -2*n**2*(n+2)*(n+3), 288, 90, -31, .27687),
 (3*n**2+9*n+7, -2*n*(n+1)**3, 1, -2, 2, .30672),
 (3*n**2+11*n+5, -2*n**3*(n+4), 24, 18, -11, .30966),
 (3*n**2+11*n+5, -2*n**2*(n+1)*(n+4), 16, 6, -1, .30103),
 (3*n**2+11*n+5, -2*n**2*(n+2)*(n+4), 64, 18, 13, .29221),
 (3*n**2+11*n+9, -2*n**2*(n+2)**2, 4, 6, -5, .30958),
 (3*n**2+11*n+9, -2*n*(n+1)*(n+2)**2, 8, -2, 3, .30095),
 (3*n**2+11*n+9, -2*n*(n+2)**3, 32, 2, 5, .29213),
 (3*n**2+11*n+9, -2*n*(n+2)**2*(n+3), 192, 18, 13, .28311),
 (3*n**2+13*n+13, -2*n*(n+1)**2*(n+3), 6, -18, 17, .31239),
 (3*n**2+15*n+15, -2*n**2*(n+2)*(n+4), 48, 90, -79, .31522),
 (3*n**2+15*n+15, -2*n*(n+1)*(n+2)*(n+4), 32, -18, 19, .30674),
 (3*n**2+15*n+15, -2*n*(n+2)**2*(n+4), 128, -6, 17, .29808),
 (3*n**2+15*n+19, -2*n*(n+2)**3, 8, 54, -49, .31515),
 (3*n**2+17*n+23, -2*n*(n+1)*(n+3)**2, 12, -90, 83, .31792),
]
assert len(rm_rows) == 22

def pcf_convergent(a, b, depth):
    # Standard generalized-CF continuants, evaluated exactly.
    pm2, pm1 = sp.Integer(1), sp.Integer(a.subs(n, 0))
    qm2, qm1 = sp.Integer(0), sp.Integer(1)
    for j in range(1, depth+1):
        aj, bj = int(a.subs(n, j)), int(b.subs(n, j))
        pm2, pm1 = pm1, aj*pm1 + bj*pm2
        qm2, qm1 = qm1, aj*qm1 + bj*qm2
    return sp.Rational(pm1, qm1)

# Convert the catalogue's fractional-linear readout to p_N/q_N for G.
def catalan_approximant(row, depth):
    a, b, K, c, d, _ = rm_rows[row - 1]
    z = pcf_convergent(a, b, depth)
    return sp.Rational(K - d*z, c*z)

depths = list(range(10, 301, 10))
approximants = [catalan_approximant(22, N) for N in depths]
deltas = [delta(r) for r in approximants]
results["Ramanujan Machine (2020), row 22"] = (depths, deltas)
accuracy["Ramanujan Machine (2020), row 22"] = [correct_digits(r) for r in approximants]
print(f"row 22: delta at depth 300: {deltas[-1]:.9f}")

# The other 21 proposed readouts are checked at depth 80.
assert all(correct_digits(catalan_approximant(row, 80)) > 10 for row in range(1, 23))
print("All 22 catalogue readouts agree with G to more than 10 digits at depth 80.")


### Two-axis CMFs for the catalogue

Put $X(n,k)=\begin{pmatrix}0&b(n,k)\\1&a(n,k)\end{pmatrix}$.
For fixed $i,j$, the **even-pair family** has $i,j$ even, vertical step
$k\mapsto k+1$, and
\[
a_g=3n^2+(2i+2j+3)n+(i+1)(j+1),\qquad
b_g=-2n(n+i)(n+j)(n+k).
\]
The **same-parity family** has $i,j,k$ of the same parity, vertical step
$k\mapsto k+2$, and
\[
a_f=3n^2+[2(i+j+k)+3]n+(i+1)(j+1)(k+1)-ijk,\qquad
b_f=-2n(n+i)(n+j)(n+k).
\]
The code constructs the vertical matrices $Y_g,Y_f$, verifies
$X(n,k)Y(n+1,k)=Y(n,k)X(n,k+h)$ exactly, and identifies every row.
This proves the CMF recurrence recovery.  It does not prove the
catalogue's proposed values of the infinite continued fractions.


In [ ]:
# Exact symbolic CMF certificate and assignment of every catalogue row.
i, j, k = sp.symbols("i j k", integer=True, nonnegative=True)
bg = -2*n*(n+i)*(n+j)*(n+k)
ag = 3*n**2+(2*i+2*j+3)*n+(i+1)*(j+1)
Xg = sp.Matrix([[0,bg],[1,ag]])
Yg = sp.Matrix([
 [4*k**2+2*k*n+2*(1-i-j)*(k+n)-2*n**2, bg],
 [1,4*k**2+2*k*n+(4-2*i-2*j)*k+n**2+n+(i-1)*(j-1)]
])
bf = bg
af = 3*n**2+(2*(i+j+k)+3)*n+(i+1)*(j+1)*(k+1)-i*j*k
Xf = sp.Matrix([[0,bf],[1,af]])
Yf = sp.Matrix([
 [(k**2+k-n**2+n)/2,bf/2],
 [sp.Rational(1,2),(k**2+2*k*n+(i+j+2)*k+2*n**2+2*(i+j+1)*n+(i+1)*(j+1))/2]
])
assert all(sp.expand(e)==0 for e in Xg*Yg.subs(n,n+1)-Yg*Xg.subs(k,k+1))
assert all(sp.expand(e)==0 for e in Xf*Yf.subs(n,n+1)-Yf*Xf.subs(k,k+2))
assert sp.expand(Yg.det()+2*(k+n)*(i-2*k-1)*(j-2*k-1)*(i+j-2*k-1)) == 0
assert sp.expand(Yf.det()-(k+n)*(i+k+1)*(j+k+1)*(k+n+1)/4) == 0

def rm_cmf_coordinates(a, b):
    # Three nonnegative offsets are read from -b/(2n).
    offsets = sorted(-int(r) for r in sp.Poly(sp.cancel(-b/(2*n)),n).all_roots())
    assert len(offsets)==3 and all(t>=0 for t in offsets)
    candidates=[]
    for ii in sorted(set(offsets)):
      for jj in sorted(set(offsets)):
       for kk in sorted(set(offsets)):
        if sorted((ii,jj,kk)) != offsets: continue
        point={i:ii,j:jj,k:kk}
        if ii%2==jj%2==0 and sp.expand(a-ag.subs(point))==0:
            candidates.append(('even-pair',ii,jj,kk))
        if ii%2==jj%2==kk%2 and sp.expand(a-af.subs(point))==0:
            candidates.append(('same-parity',ii,jj,kk))
    assert candidates, (a,b)
    return sorted(set(candidates), key=lambda t: (t[0] != "even-pair", t[1:]))[0]

rm_cmf_rows=[rm_cmf_coordinates(a,b) for a,b,*_ in rm_rows]
assert len(rm_cmf_rows)==22
print(f"Exact flatness verified for both fields; {len(rm_cmf_rows)} catalogue rows recovered.")
for row, coordinates in enumerate(rm_cmf_rows, 1):
    print(f"{row:2}: {coordinates[0]:11}  (i,j,k)={coordinates[1:]}")


## All constructions

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## $\delta_N$ against accuracy digits

The same samples against $D_N$ instead of $N$, as in the paper's figure of
$\delta_N$ against accuracy digits.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(accuracy[name], values, label=name)
ax.set(xlabel="$D_N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Viola–Marcovecchio (2022): reference only

Paper Table 1 includes [their reported error power](https://doi.org/10.4171/OWR/2022/21)
$0.6293$, corresponding to a source-construction exponent at least $-0.3707$.
There is no verified CMF encoding or computed finite-depth $\delta_N$ for
this construction in this notebook. The paper table leaves those cells blank.


## Historical overview

| Construction | Reported | Here |
|:--|--:|:--|
| Zudilin original (2002/2003) | derived $\liminf\delta_N\ge-0.4757268902$ | five-axis CMF; $\delta_{1000}=-0.473017704448925$ |
| Zudilin modified (2002) | same derived lower bound | five-axis CMF; $\delta_{1000}=-0.472536570512580$ |
| Nesterenko (2016) | — | CMF |
| [Viola–Marcovecchio (2022)](https://doi.org/10.4171/OWR/2022/21) | error power $0.6293$, $\delta\ge-0.3707$ | reference only |
| Ramanujan Machine catalogue (2020) | 22 conjectured PCFs; speeds are digits/term | two-axis CMF and finite-depth $\delta_N$ above |
| Ramanujan Machine (2021), SI Tables 5–6 | eight examples; Table 6 includes reported error power about $0.567$ | source set identified, CMF matching open |
| Eskandari (2026) | $\kappa\ge0.62$ | CMF |
| This work, ${}_3F_2(1)$ direction search | — | CMF |
| This work, ${}_4F_3(1)$ | proved $\delta>-0.295976878$ | CMF |